# Recommender Metrics Evaluation

Export MovieLens recommender metrics to `reports/metrics_recommender.csv`.


In [ ]:
import json
import pandas as pd
from pathlib import Path

metrics_path = Path('experiments') / 'recommender' / 'metrics' / 'movielens_metrics.json'
rows = []
if metrics_path.exists():
    report = json.loads(metrics_path.read_text(encoding='utf-8'))
    if 'accuracy' in report:
        rows.append({'Metric': 'accuracy', 'mean': float(report['accuracy']), 'std': ''})
    macro = report.get('macro avg', {})
    if macro:
        rows.extend([
            {'Metric': 'precision', 'mean': float(macro.get('precision', 0.0)), 'std': ''},
            {'Metric': 'recall', 'mean': float(macro.get('recall', 0.0)), 'std': ''},
            {'Metric': 'f1', 'mean': float(macro.get('f1-score', 0.0)), 'std': ''},
        ])

report_path = Path('reports') / 'metrics_recommender.csv'
df_new = pd.DataFrame(rows)
if report_path.exists():
    df_old = pd.read_csv(report_path)
    if 'std' not in df_old.columns:
        df_old['std'] = ''
    df_old = df_old[~df_old['Metric'].isin(df_new['Metric'])]
    df_out = pd.concat([df_old, df_new], ignore_index=True)
else:
    df_out = df_new
report_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(report_path, index=False)
print('Saved metrics to', report_path)
